# NLP Parse Lineage & Ambiguity Audit Recipe

This recipe combines 3 `algebrax` tools to audit natural language syntax trees:

1. **Matrix CYK Parsing** (`algebrax.matrix.core.dot`):
   Executes Chomsky Normal Form (CNF) grammar parsing via matrix multiplication (`dot`).
2. **Symbolic Rule Provenance** (`algebrax.semiring.ProvenanceSemiring`):
   Tracks symbolic rule derivation polynomials ($X[r_1, r_2, \dots]$).
3. **Structural Entropy Audit** (`algebrax.probability.entropy`):
   Quantifies parse ambiguity across candidate parse trees.

In [ ]:
import algebrax as ax


class GrammarSemiring(ax.semiring.Semiring[set[str]]):
    def __init__(self, rules: dict[tuple[str, str], set[str]]):
        self.rules = rules

    @property
    def zero(self) -> set[str]:
        return set()

    @property
    def one(self) -> set[str]:
        return set()

    def add(self, a: set[str], b: set[str]) -> set[str]:
        return a | b

    def mul(self, a: set[str], b: set[str]) -> set[str]:
        res = set()
        for nt1 in a:
            for nt2 in b:
                res |= self.rules.get((nt1, nt2), set())
        return res

## 1. Matrix CYK Parsing via Dot Product

We parse the target sentence `'the astronomer saw stars'` using matrix multiplication over `GrammarSemiring`.

In [ ]:
import algebrax as ax

sentence = ['the', 'astronomer', 'saw', 'stars']
lexicon = {'the': {'Det'}, 'astronomer': {'N', 'NP'}, 'saw': {'V'}, 'stars': {'N', 'NP'}}
grammar_rules = {('Det', 'N'): {'NP'}, ('V', 'NP'): {'VP'}, ('NP', 'VP'): {'S'}}

grammar_semiring = GrammarSemiring(grammar_rules)
n_len = len(sentence)
chart = {}
for i, word in enumerate(sentence):
    chart[i] = {i + 1: lexicon.get(word, set())}

for _ in range(n_len):
    new_spans = ax.matrix.dot(chart, chart, semiring=grammar_semiring)
    for r, row in new_spans.items():
        if r not in chart:
            chart[r] = {}
        for c, val in row.items():
            chart[r][c] = chart[r].get(c, set()) | val

print(f'Parsed Full Sentence Non-Terminals: {chart.get(0, {}).get(n_len, set())}')

## 2. Symbolic Rule Derivation Provenance

We compute derivation polynomials over `ProvenanceSemiring`.

In [ ]:
import algebrax as ax

provenance_semiring = ax.semiring.ProvenanceSemiring()
rule_x = {('rule_DetN_to_NP',): 1}
rule_y = {('rule_VNP_to_VP',): 1}
rule_z = {('rule_NPVP_to_S',): 1}

derivation = provenance_semiring.mul(provenance_semiring.mul(rule_x, rule_y), rule_z)
print('Symbolic Rule Lineage Polynomial:')
for terms, coeff in derivation.items():
    print(f'  Coeff {coeff}: {" * ".join(terms)}')

## 3. Structural Parse Ambiguity & Entropy

We calculate Shannon entropy $H(P) = - \sum p_i \ln p_i$ to quantify parse ambiguity.

In [ ]:
import algebrax as ax

candidate_parse_probs = {
    'Parse_Tree_Direct_Object': 0.75,
    'Parse_Tree_Prepositional_Attachment': 0.15,
    'Parse_Tree_Noun_Compound': 0.10,
}

parse_entropy = ax.probability.entropy(candidate_parse_probs)
print(f'Parse Tree Structural Entropy H(Trees): {parse_entropy:.4f} nats')